In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd

from PortfolioOptimization import get_report

In [2]:
max_risk = 0.15

In [3]:
output_dir = './Output/Change in Max Risk'
os.makedirs(output_dir, exist_ok=True)
output_dir = f'{output_dir}/{max_risk:.2%}'
os.makedirs(output_dir, exist_ok=True)

In [4]:
from Portfolios.GSPCBuyAndHold import BuyAndHold
from Portfolios.EqualWeights import EqWeightPortfolio
from Portfolios.Markowitz import MarkowitzPortfolio
from Portfolios.MeanAbsDev import MeanAbsDevPortfolio

freqs = [
    # '1d',
    # '1wk',
    '1mo',
    '3mo',
    '6mo',
    '1y'
]
portfolios = {
    'S&P BnH': BuyAndHold,
    'Equal': EqWeightPortfolio,
    'Markowitz': MarkowitzPortfolio,
    'MAD': MeanAbsDevPortfolio,
}

reports = {}

In [5]:
for freq in freqs:
    os.makedirs(f'{output_dir}/{freq}', exist_ok=True)
    os.makedirs(f'{output_dir}/{freq}/Graphs', exist_ok=True)
    all_stats = {}
    for name, portfolio in portfolios.items():
        weights, stats = get_report(portfolio, freq=freq, max_risk=max_risk)
        weights.to_excel(f'{output_dir}/{freq}/{name}.xlsx')

        all_stats[name] = stats
        stats_used = list(stats.columns)

    for stat in stats_used:
        stat_graph_df = pd.concat([
                            all_stats[name][[stat]].rename(columns={stat: name})
                            for name in portfolios.keys()
                        ], axis=1)
        if stat == 'PV':
            stat_graph_df.plot(kind='line', figsize=(18, 6))
        else:
            stat_graph_df.plot(kind='bar', figsize=(18, 6))
        plt.xticks(rotation=90)
        plt.title(f"{stat.title()} Comparison for all Portfolios (Rebalanced {freq})")
        plt.savefig(f'{output_dir}/{freq}/Graphs/{stat}.png')
        plt.close()